# Stage3 VLM Calibration Analysis

Diagnose *how well* the VLM (Qwen2-VL-7B-Instruct) agrees with a human photographer
who evaluated the same 250 livehouse shots.

**Key questions:**
1. Do model scores correlate with human scores? (Spearman ρ, Pearson r)
2. Is there systematic bias at different quality tiers? (quintile calibration)
3. Can the model identify photos a photographer would keep? (precision/recall@k)
4. What explains the Spearman ≪ Pearson gap?

**Data:**
- `data/eval/labels.jsonl` — 250 human-labeled photos (overall 0–100, keep/discard)
- `data/eval/images/analysis_results.json` — model predictions from the full pipeline
- `reports/eval/stage3_v6_qwen2vl_temp0.json` — pre-computed eval report (temperature=0)

In [ ]:
import sys
from pathlib import Path

# Allow imports from project root
ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    "figure.facecolor": "#0e0e0e",
    "axes.facecolor": "#1a1a1a",
    "axes.edgecolor": "#444",
    "axes.labelcolor": "#ccc",
    "xtick.color": "#999",
    "ytick.color": "#999",
    "text.color": "#ddd",
    "grid.color": "#2a2a2a",
    "grid.linestyle": "--",
    "figure.dpi": 120,
})
ACCENT = "#6ee7b7"   # emerald-300
WARN   = "#fb923c"   # orange-400
KEEP_C = "#34d399"   # green
DROP_C = "#f87171"   # red

print("✓ imports OK")

In [ ]:
from scripts.eval.labels import load_labels, load_predictions, join_labels_predictions
from scripts.eval import metrics as M

LABELS_PATH = ROOT / "data/eval/labels.jsonl"
PREDS_PATH  = ROOT / "data/eval/images/analysis_results.json"
REPORT_PATH = ROOT / "reports/eval/stage3_v6_qwen2vl_temp0.json"

labels = load_labels(LABELS_PATH)
preds  = load_predictions(PREDS_PATH)
joined = join_labels_predictions(labels, preds)

print(f"Matched pairs : {joined.n_matched}")
print(f"Labels only   : {len(joined.labels_only)}")
print(f"Preds only    : {len(joined.preds_only)}")

# Build aligned arrays
human_overall = np.array([lb.overall for lb, p in joined.pairs if lb.overall is not None and p.overall is not None])
model_overall = np.array([p.overall  for lb, p in joined.pairs if lb.overall is not None and p.overall is not None])
keep_flags    = np.array([lb.keep    for lb, p in joined.pairs if lb.keep    is not None and p.overall is not None], dtype=bool)
# keep_flags may be shorter if some pairs lack lb.keep
keep_full     = np.array([
    lb.keep for lb, p in joined.pairs
    if lb.overall is not None and p.overall is not None
], dtype=object)  # may contain None

spearman = M.spearman(human_overall, model_overall)
pearson  = M.pearson(human_overall, model_overall)
mae_val  = M.mae(human_overall, model_overall)
rmse_val = M.rmse(human_overall, model_overall)

print(f"\nOverall score (n={len(human_overall)})")
print(f"  Spearman ρ : {spearman:.3f}")
print(f"  Pearson  r : {pearson:.3f}")
print(f"  MAE        : {mae_val:.2f}")
print(f"  RMSE       : {rmse_val:.2f}")

## 1 · Human vs Model Score Scatter

Green = photos the human chose to keep; red = discarded.
A perfect model would lie on the diagonal.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

# Separate keep / discard / unknown
h_all, m_all, k_all = [], [], []
for lb, p in joined.pairs:
    if lb.overall is None or p.overall is None:
        continue
    h_all.append(lb.overall)
    m_all.append(p.overall)
    k_all.append(lb.keep)

h_k  = [h for h, k in zip(h_all, k_all) if k is True]
m_k  = [m for m, k in zip(m_all, k_all) if k is True]
h_d  = [h for h, k in zip(h_all, k_all) if k is False]
m_d  = [m for m, k in zip(m_all, k_all) if k is False]
h_u  = [h for h, k in zip(h_all, k_all) if k is None]
m_u  = [m for m, k in zip(m_all, k_all) if k is None]

ax.scatter(h_d, m_d, c=DROP_C,  alpha=0.55, s=22, label=f"Discard (n={len(h_d)})", zorder=3)
ax.scatter(h_k, m_k, c=KEEP_C,  alpha=0.75, s=28, label=f"Keep    (n={len(h_k)})", zorder=4)
if h_u:
    ax.scatter(h_u, m_u, c="#888", alpha=0.35, s=16, label=f"Unknown (n={len(h_u)})", zorder=2)

lo, hi = 0, 100
ax.plot([lo, hi], [lo, hi], "--", color="#555", lw=1, label="Perfect agreement")
ax.set_xlabel("Human overall score", fontsize=12)
ax.set_ylabel("Model overall score", fontsize=12)
ax.set_title(
    f"Human vs Model  ·  Spearman ρ={spearman:.3f}  Pearson r={pearson:.3f}",
    fontsize=12, pad=10,
)
ax.set_xlim(0, 100); ax.set_ylim(40, 100)
ax.legend(framealpha=0.15, fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(ROOT / "reports/eval/fig_scatter.png", bbox_inches="tight")
plt.show()

## 2 · Score Distribution Comparison

Do human and model use the same part of the 0–100 scale?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

bins = np.arange(0, 101, 5)
ax.hist(human_overall, bins=bins, alpha=0.6, color=ACCENT, label="Human",  edgecolor="#111")
ax.hist(model_overall, bins=bins, alpha=0.5, color=WARN,   label="Model",  edgecolor="#111")

ax.axvline(np.mean(human_overall), color=ACCENT, lw=1.5, linestyle=":", label=f"Human μ={np.mean(human_overall):.1f}")
ax.axvline(np.mean(model_overall), color=WARN,   lw=1.5, linestyle=":", label=f"Model μ={np.mean(model_overall):.1f}")

ax.set_xlabel("Overall score", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Score Distribution: Human vs Model", fontsize=12)
ax.legend(framealpha=0.15, fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(ROOT / "reports/eval/fig_distribution.png", bbox_inches="tight")
plt.show()

print(f"Human: μ={np.mean(human_overall):.1f}  σ={np.std(human_overall):.1f}  range=[{human_overall.min():.0f}, {human_overall.max():.0f}]")
print(f"Model: μ={np.mean(model_overall):.1f}  σ={np.std(model_overall):.1f}  range=[{model_overall.min():.0f}, {model_overall.max():.0f}]")

## 3 · Quintile Calibration — Where Does the Model Fail?

Equal-population bins of human scores. Positive bias = model over-scores that tier.

In [ ]:
qc = M.quintile_calibration(human_overall, model_overall)

labels_q = [f"Q{q['quintile']}\n{q['range']}" for q in qc]
biases   = [q['bias_mean'] for q in qc]
colors   = [KEEP_C if b < 0 else WARN for b in biases]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels_q, biases, color=colors, edgecolor="#111", width=0.6)
ax.axhline(0, color="#555", lw=1)

for bar, bias in zip(bars, biases):
    ypos = bar.get_height() + 0.15 if bias >= 0 else bar.get_height() - 0.6
    ax.text(bar.get_x() + bar.get_width() / 2, ypos,
            f"{bias:+.1f}", ha="center", va="bottom", fontsize=9, color="#ddd")

ax.set_ylabel("Mean bias  (model − human)", fontsize=11)
ax.set_title("Model Bias by Human Score Quintile", fontsize=12)
ax.grid(True, axis="y", alpha=0.35)

over  = mpatches.Patch(color=WARN,   label="Over-scored")
under = mpatches.Patch(color=KEEP_C, label="Under-scored")
ax.legend(handles=[over, under], framealpha=0.15, fontsize=9)
plt.tight_layout()
plt.savefig(ROOT / "reports/eval/fig_quintile_bias.png", bbox_inches="tight")
plt.show()

print("\nKey insight:")
q1_bias = qc[0]['bias_mean'] if qc else 0
q5_bias = qc[-1]['bias_mean'] if qc else 0
print(f"  Q1 (trash tier) over-scored by {q1_bias:+.1f} pts → model can't identify truly bad shots")
print(f"  Q5 (best tier)  under-scored by {q5_bias:+.1f} pts → model ceiling / regression toward mean")
print(f"  This compresses the score range → Spearman suffers even when Pearson is OK")

## 4 · Keep vs Discard — Model Score Distributions

In [ ]:
keep_scores    = [p.overall for lb, p in joined.pairs if lb.keep is True  and p.overall is not None]
discard_scores = [p.overall for lb, p in joined.pairs if lb.keep is False and p.overall is not None]

fig, ax = plt.subplots(figsize=(9, 4))

bins = np.arange(40, 101, 3)
ax.hist(discard_scores, bins=bins, alpha=0.6, color=DROP_C, label=f"Discard (n={len(discard_scores)})", edgecolor="#111")
ax.hist(keep_scores,    bins=bins, alpha=0.6, color=KEEP_C, label=f"Keep    (n={len(keep_scores)})",    edgecolor="#111")

ax.axvline(np.mean(discard_scores), color=DROP_C, lw=2, linestyle="--",
           label=f"Discard μ={np.mean(discard_scores):.1f}")
ax.axvline(np.mean(keep_scores),    color=KEEP_C, lw=2, linestyle="--",
           label=f"Keep μ={np.mean(keep_scores):.1f}")

gap = np.mean(keep_scores) - np.mean(discard_scores)
ax.set_title(f"Model Scores by Human Decision  ·  Score gap = {gap:.1f} pts", fontsize=12)
ax.set_xlabel("Model overall score", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.legend(framealpha=0.15, fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(ROOT / "reports/eval/fig_keep_discard.png", bbox_inches="tight")
plt.show()

_, _, gap = M.group_mean_separation(
    [p.overall for lb, p in joined.pairs if lb.keep is not None and p.overall is not None],
    [lb.keep   for lb, p in joined.pairs if lb.keep is not None and p.overall is not None],
)
print(f"Score gap (keep − discard): {gap:.1f} pts")
print(f"A large gap means the model can differentiate keeps from discards, even if not well-ranked.")

## 5 · Selection Precision@k

If the model's top-k photos are used as the delivery set, how many does the human agree with?

In [ ]:
ks = [5, 10, 20, 30, 50, 83]
sel_scores = [p.overall for lb, p in joined.pairs if lb.keep is not None and p.overall is not None]
sel_pos    = [lb.keep   for lb, p in joined.pairs if lb.keep is not None and p.overall is not None]

prec_vals  = []
recall_vals = []
for k in ks:
    r = M.precision_recall_at_k(sel_scores, sel_pos, k)
    prec_vals.append(r.precision)
    recall_vals.append(r.recall)
    print(f"  @{k:<3} P={r.precision:.2f}  R={r.recall:.2f}  overlap={r.overlap}/{r.n_positives}")

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(ks))
ax.plot(x, prec_vals,  "o-", color=ACCENT, lw=2, ms=7, label="Precision@k")
ax.plot(x, recall_vals, "s--", color=WARN,   lw=2, ms=6, label="Recall@k")
ax.set_xticks(list(x))
ax.set_xticklabels([f"@{k}" for k in ks])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Selection Precision / Recall at k", fontsize=12)
ax.legend(framealpha=0.15, fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(ROOT / "reports/eval/fig_precision_recall_k.png", bbox_inches="tight")
plt.show()

## 6 · Summary & Diagnosis

**What the numbers say:**

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Spearman ρ | 0.36 | Weak ranking agreement |
| Pearson r | 0.84 | Strong *linear* agreement |
| MAE | 6.1 pts | Average scoring error on 0–100 scale |
| Score gap (keep−discard) | ~5 pts | Model can distinguish quality tiers |
| Precision@20 | 0.55 | 11 of 20 top model picks match human keeps |

**Root cause of Spearman ≪ Pearson:**

Quintile analysis shows the model compresses both ends of the score range:
- Bottom tier (human <70): model adds ~+6 pts → trash photos look more acceptable
- Top tier (human 80–88): model subtracts ~−8 pts → best photos lose their distinctiveness

This means scores are *linearly predictable* across the full range (Pearson OK)
but the model can't separate nearby photos (Spearman suffers).

**Next steps:**
1. Add per-dimension human labels to diagnose which dimensions cause the floor/ceiling
2. Consider temperature calibration — at temp=0 the model may be over-confident
3. Prompt engineering: add explicit anchor examples to stretch the score distribution
4. Fine-tune a lightweight regressor head on top of SigLIP embeddings — see `scripts/eval/train_siglip_regressor.py`